# LegisAnalytica — Gold Layer DDL

Projeto conceitual e lógico → implementação física no schema `workspace.pi_ii_gold`.

**Fonte:** Modelo relacional LegisAnalytica v5 (PI II • Grupo 14 • Univesp)  
**Origem dos dados:** 13 tabelas Silver em `workspace.pi_ii_silver`

## 26 tabelas do núcleo relacional

| Grupo | Tabelas |
|-------|---------|
| Catálogos | CASA, UF, LEGISLATURA, PARTIDO, LOTE, CAUSA_AFASTAMENTO |
| Identidade & Exercício | PARLAMENTAR, IDENTIFICADOR_PARLAMENTAR, CADASTRO_OBSERVADO, PARLAMENTAR_LEGISLATURA, MANDATO, VINCULO_MANDATO, EXERCICIO |
| Atividade Legislativa | TIPO_PROPOSICAO, PROPOSICAO, AUTORIA, STATUS_ATUAL, VOTACAO, VOTACAO_PROPOSICAO, TIPO_VOTO, VOTO, TOTAL_PUBLICADO |
| Despesas | TIPO_DESPESA, FORNECEDOR, DESPESA, VALOR_DESPESA |

## Convenções
- `?` = opcional (NULL permitido); demais campos são obrigatórios
- `PK` = chave primária; `FK` = chave estrangeira; `UQ` = chave alternativa
- `casa ∈ {CAMARA, SENADO}`
- IDs são texto (STRING) conforme domínio do projeto
- Zero em numero/ano de origem é convertido em nulo

In [0]:
%sql
-- ============================================================
-- TABELAS 01-06: LOTE + Catálogos (CASA, UF, LEGISLATURA, PARTIDO, CAUSA_AFASTAMENTO)
-- ============================================================

-- 05 LOTE — um lote por entidade de origem Silver
-- PK(id_lote); entidade_fonte; localizador; hash_sha256; processado_en; coletado_en?; cobertura_inicio?; cobertura_fim?
-- UQ(entidade_fonte, hash_sha256)
CREATE OR REPLACE TABLE workspace.pi_ii_gold.lote AS
SELECT
  entidade_fonte AS id_lote,
  entidade_fonte AS entidade_fonte,
  CONCAT('workspace.pi_ii_silver.', entidade_fonte) AS localizador,
  sha2(entidade_fonte, 256) AS hash_sha256,
  current_timestamp() AS processado_em,
  CAST(NULL AS TIMESTAMP) AS coletado_em,
  CAST(NULL AS DATE) AS cobertura_inicio,
  CAST(NULL AS DATE) AS cobertura_fim
FROM (
  SELECT 'deputados' AS entidade_fonte
  UNION ALL SELECT 'senadores'
  UNION ALL SELECT 'senadores_exercicios'
  UNION ALL SELECT 'proposicoes'
  UNION ALL SELECT 'proposicoes_autores'
  UNION ALL SELECT 'materias'
  UNION ALL SELECT 'materias_autorias'
  UNION ALL SELECT 'votacoes_camara'
  UNION ALL SELECT 'votacoes_senado'
  UNION ALL SELECT 'votos_camara'
  UNION ALL SELECT 'votos_senado'
  UNION ALL SELECT 'despesas_camara'
  UNION ALL SELECT 'despesas_senado'
) t;

-- 01 CASA
-- PK(casa); nome
CREATE OR REPLACE TABLE workspace.pi_ii_gold.casa AS
SELECT 'CAMARA' AS casa, 'Câmara dos Deputados' AS nome
UNION ALL
SELECT 'SENADO', 'Senado Federal';

-- 02 UF — catálogo das 27 unidades federativas
-- PK(uf); nome
CREATE OR REPLACE TABLE workspace.pi_ii_gold.uf AS
SELECT 'AC' AS uf, 'Acre' AS nome
UNION ALL SELECT 'AL', 'Alagoas'
UNION ALL SELECT 'AP', 'Amapá'
UNION ALL SELECT 'AM', 'Amazonas'
UNION ALL SELECT 'BA', 'Bahia'
UNION ALL SELECT 'CE', 'Ceará'
UNION ALL SELECT 'DF', 'Distrito Federal'
UNION ALL SELECT 'ES', 'Espírito Santo'
UNION ALL SELECT 'GO', 'Goiás'
UNION ALL SELECT 'MA', 'Maranhão'
UNION ALL SELECT 'MT', 'Mato Grosso'
UNION ALL SELECT 'MS', 'Mato Grosso do Sul'
UNION ALL SELECT 'MG', 'Minas Gerais'
UNION ALL SELECT 'PA', 'Pará'
UNION ALL SELECT 'PB', 'Paraíba'
UNION ALL SELECT 'PR', 'Paraná'
UNION ALL SELECT 'PE', 'Pernambuco'
UNION ALL SELECT 'PI', 'Piauí'
UNION ALL SELECT 'RJ', 'Rio de Janeiro'
UNION ALL SELECT 'RN', 'Rio Grande do Norte'
UNION ALL SELECT 'RS', 'Rio Grande do Sul'
UNION ALL SELECT 'RO', 'Rondônia'
UNION ALL SELECT 'RR', 'Roraima'
UNION ALL SELECT 'SC', 'Santa Catarina'
UNION ALL SELECT 'SP', 'São Paulo'
UNION ALL SELECT 'SE', 'Sergipe'
UNION ALL SELECT 'TO', 'Tocantins';

-- 03 LEGISLATURA
-- PK(numero_legislatura); inicio; fim
CREATE OR REPLACE TABLE workspace.pi_ii_gold.legislatura AS
SELECT 57 AS numero_legislatura, DATE '2023-02-01' AS inicio, DATE '2027-01-31' AS fim;

-- 04 PARTIDO — união das siglas observadas na Silver
-- PK(sigla)
CREATE OR REPLACE TABLE workspace.pi_ii_gold.partido AS
SELECT DISTINCT partido AS sigla
FROM (
  SELECT partido FROM workspace.pi_ii_silver.deputados WHERE partido IS NOT NULL AND partido != ''
  UNION
  SELECT partido FROM workspace.pi_ii_silver.senadores WHERE partido IS NOT NULL AND partido != ''
  UNION
  SELECT partido_autor AS partido FROM workspace.pi_ii_silver.proposicoes_autores WHERE partido_autor IS NOT NULL AND partido_autor != ''
  UNION
  SELECT partido FROM workspace.pi_ii_silver.votos_camara WHERE partido IS NOT NULL AND partido != ''
  UNION
  SELECT partido FROM workspace.pi_ii_silver.votos_senado WHERE partido IS NOT NULL AND partido != ''
  UNION
  SELECT partido FROM workspace.pi_ii_silver.despesas_camara WHERE partido IS NOT NULL AND partido != ''
) t
WHERE partido IS NOT NULL AND partido != '';

-- 12 CAUSA_AFASTAMENTO — do Senado
-- PK(casa, codigo_causa); descricao
CREATE OR REPLACE TABLE workspace.pi_ii_gold.causa_afastamento AS
SELECT
  'SENADO' AS casa,
  sigla_causa_afastamento AS codigo_causa,
  MAX(descricao_causa_afastamento) AS descricao
FROM workspace.pi_ii_silver.senadores_exercicios
WHERE sigla_causa_afastamento IS NOT NULL AND sigla_causa_afastamento != ''
GROUP BY sigla_causa_afastamento;

In [0]:
%sql
-- ============================================================
-- TABELAS 06-09: Identidade parlamentar
-- ============================================================

-- 06 PARLAMENTAR — cadastro de uma Casa
-- PK(casa, id_parlamentar); nome_referencia; url_foto?; email?; id_lote
CREATE OR REPLACE TABLE workspace.pi_ii_gold.parlamentar AS
SELECT
  'CAMARA' AS casa,
  CAST(id_deputado AS STRING) AS id_parlamentar,
  nome AS nome_referencia,
  url_foto,
  email,
  'deputados' AS id_lote
FROM workspace.pi_ii_silver.deputados
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_senador AS STRING) AS id_parlamentar,
  nome AS nome_referencia,
  CAST(NULL AS STRING) AS url_foto,
  CAST(NULL AS STRING) AS email,
  'senadores' AS id_lote
FROM workspace.pi_ii_silver.senadores;

-- 07 IDENTIFICADOR_PARLAMENTAR — código de um sistema
-- PK(casa, sistema, codigo_externo); id_parlamentar?
-- FK(casa, id_parlamentar) → PARLAMENTAR. sistema ∈ {API, CEAP}
CREATE OR REPLACE TABLE workspace.pi_ii_gold.identificador_parlamentar AS
-- Códigos API já resolvidos dos cadastros (Câmara)
SELECT
  'CAMARA' AS casa,
  'API' AS sistema,
  CAST(id_deputado AS STRING) AS codigo_externo,
  CAST(id_deputado AS STRING) AS id_parlamentar
FROM workspace.pi_ii_silver.deputados
UNION ALL
-- Códigos API já resolvidos dos cadastros (Senado)
SELECT
  'SENADO' AS casa,
  'API' AS sistema,
  CAST(id_senador AS STRING) AS codigo_externo,
  CAST(id_senador AS STRING) AS id_parlamentar
FROM workspace.pi_ii_silver.senadores
UNION ALL
-- Códigos CEAP das despesas da Câmara (mapeados por nome → API)
SELECT DISTINCT
  'CAMARA' AS casa,
  'CEAP' AS sistema,
  CAST(dc.id_deputado AS STRING) AS codigo_externo,
  CAST(d.id_deputado AS STRING) AS id_parlamentar
FROM workspace.pi_ii_silver.despesas_camara dc
LEFT JOIN workspace.pi_ii_silver.deputados d
  ON UPPER(dc.nome_parlamentar) = UPPER(d.nome)
WHERE dc.id_deputado IS NOT NULL;

-- 08 CADASTRO_OBSERVADO — fotografia por parlamentar e lote
-- PK(casa, id_parlamentar, id_lote); partido?; uf?
-- FK(casa, id_parlamentar) → PARLAMENTAR
CREATE OR REPLACE TABLE workspace.pi_ii_gold.cadastro_observado AS
SELECT
  'CAMARA' AS casa,
  CAST(id_deputado AS STRING) AS id_parlamentar,
  'deputados' AS id_lote,
  partido,
  uf
FROM workspace.pi_ii_silver.deputados
WHERE partido IS NOT NULL OR uf IS NOT NULL
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_senador AS STRING) AS id_parlamentar,
  'senadores' AS id_lote,
  partido,
  uf
FROM workspace.pi_ii_silver.senadores
WHERE partido IS NOT NULL OR uf IS NOT NULL;

-- 09 PARLAMENTAR_LEGISLATURA — membro da coorte
-- PK(casa, id_parlamentar, numero_legislatura); criterio_inclusao; id_lote
-- FK(casa, id_parlamentar) → PARLAMENTAR; numero_legislatura → LEGISLATURA
-- Câmara: critério id_legislatura = 57; Senado: exercício intersecta a legislatura
CREATE OR REPLACE TABLE workspace.pi_ii_gold.parlamentar_legislatura AS
SELECT
  'CAMARA' AS casa,
  CAST(id_deputado AS STRING) AS id_parlamentar,
  57 AS numero_legislatura,
  'CADASTRO_FONTE' AS criterio_inclusao,
  'deputados' AS id_lote
FROM workspace.pi_ii_silver.deputados
WHERE id_legislatura = 57
UNION ALL
SELECT DISTINCT
  'SENADO' AS casa,
  CAST(se.id_senador AS STRING) AS id_parlamentar,
  57 AS numero_legislatura,
  'EXERCICIO_INTERSECTA' AS criterio_inclusao,
  'senadores_exercicios' AS id_lote
FROM workspace.pi_ii_silver.senadores_exercicios se
WHERE se.data_inicio_exercicio < DATE '2027-01-31'
  AND (se.data_fim_exercicio IS NULL OR se.data_fim_exercicio > DATE '2023-02-01');

In [0]:
%sql
-- ============================================================
-- TABELAS 10-13: Mandato e exercício (Senado)
-- ============================================================

-- 10 MANDATO — cadeira identificada
-- PK(casa, id_mandato); uf; id_lote
CREATE OR REPLACE TABLE workspace.pi_ii_gold.mandato AS
SELECT DISTINCT
  'SENADO' AS casa,
  CAST(id_mandato AS STRING) AS id_mandato,
  uf,
  'senadores_exercicios' AS id_lote
FROM workspace.pi_ii_silver.senadores_exercicios
WHERE id_mandato IS NOT NULL;

-- 11 VINCULO_MANDATO — pessoa no mandato
-- PK(casa, id_mandato, id_parlamentar); participacao; id_lote
-- FK(casa, id_mandato) → MANDATO; FK(casa, id_parlamentar) → PARLAMENTAR
CREATE OR REPLACE TABLE workspace.pi_ii_gold.vinculo_mandato AS
SELECT DISTINCT
  'SENADO' AS casa,
  CAST(id_mandato AS STRING) AS id_mandato,
  CAST(id_senador AS STRING) AS id_parlamentar,
  participacao,
  'senadores_exercicios' AS id_lote
FROM workspace.pi_ii_silver.senadores_exercicios
WHERE id_mandato IS NOT NULL AND id_senador IS NOT NULL;

-- 13 EXERCICIO — intervalo do vínculo
-- PK(casa, id_exercicio); id_mandato; id_parlamentar; inicio; fim?; codigo_causa?; id_lote
-- FK(casa, id_mandato, id_parlamentar) → VINCULO_MANDATO; FK(casa, codigo_causa) → CAUSA_AFASTAMENTO
CREATE OR REPLACE TABLE workspace.pi_ii_gold.exercicio AS
SELECT
  'SENADO' AS casa,
  CAST(id_exercicio AS STRING) AS id_exercicio,
  CAST(id_mandato AS STRING) AS id_mandato,
  CAST(id_senador AS STRING) AS id_parlamentar,
  data_inicio_exercicio AS inicio,
  data_fim_exercicio AS fim,
  sigla_causa_afastamento AS codigo_causa,
  'senadores_exercicios' AS id_lote
FROM workspace.pi_ii_silver.senadores_exercicios
WHERE id_exercicio IS NOT NULL;

In [0]:
%sql
-- ============================================================
-- TABELAS 14-18: Atividade Legislativa (TIPO_PROPOSICAO, PROPOSICAO, AUTORIA, STATUS_ATUAL, VOTACAO)
-- ============================================================

-- 14 TIPO_PROPOSICAO — catálogo de tipos
-- PK(sigla_tipo); descricao?
CREATE OR REPLACE TABLE workspace.pi_ii_gold.tipo_proposicao AS
SELECT sigla_tipo, MAX(descricao) AS descricao
FROM (
  SELECT sigla_tipo, descricao_tipo AS descricao
  FROM workspace.pi_ii_silver.proposicoes
  WHERE sigla_tipo IS NOT NULL AND sigla_tipo != ''
  UNION ALL
  SELECT sigla AS sigla_tipo, CAST(NULL AS STRING) AS descricao
  FROM workspace.pi_ii_silver.materias
  WHERE sigla IS NOT NULL AND sigla != ''
) t
GROUP BY sigla_tipo;

-- 15 PROPOSICAO — matéria ou proposição
-- PK(casa, id_proposicao); sigla_tipo; numero?; ano?; ementa?; data_apresentacao?; url_inteiro_teor?; id_lote
-- FK(sigla_tipo) → TIPO_PROPOSICAO
-- Zero em numero/ano → nulo
CREATE OR REPLACE TABLE workspace.pi_ii_gold.proposicao AS
SELECT
  'CAMARA' AS casa,
  CAST(id_proposicao AS STRING) AS id_proposicao,
  sigla_tipo,
  NULLIF(numero, 0) AS numero,
  NULLIF(ano_proposicao, 0) AS ano,
  ementa,
  data_apresentacao,
  url_inteiro_teor,
  'proposicoes' AS id_lote
FROM workspace.pi_ii_silver.proposicoes
WHERE id_proposicao IS NOT NULL
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_materia AS STRING) AS id_proposicao,
  sigla AS sigla_tipo,
  NULLIF(numero, 0) AS numero,
  NULLIF(ano_materia, 0) AS ano,
  ementa,
  data_materia AS data_apresentacao,
  CAST(NULL AS STRING) AS url_inteiro_teor,
  'materias' AS id_lote
FROM workspace.pi_ii_silver.materias
WHERE id_materia IS NOT NULL
  AND (registro_valido_leg57 IS NULL OR registro_valido_leg57 = true);

-- 16 AUTORIA — autor de uma proposição
-- PK(casa, id_proposicao, id_autor); nome_autor?; partido?; uf?; ordem_assinatura?; id_lote
-- FK(casa, id_proposicao) → PROPOSICAO
CREATE OR REPLACE TABLE workspace.pi_ii_gold.autoria AS
SELECT
  'CAMARA' AS casa,
  CAST(id_proposicao AS STRING) AS id_proposicao,
  CAST(id_deputado_autor AS STRING) AS id_autor,
  nome_autor,
  partido_autor AS partido,
  uf_autor AS uf,
  ordem_assinatura,
  'proposicoes_autores' AS id_lote
FROM workspace.pi_ii_silver.proposicoes_autores
WHERE id_proposicao IS NOT NULL AND id_deputado_autor IS NOT NULL
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_materia AS STRING) AS id_proposicao,
  CAST(id_senador AS STRING) AS id_autor,
  CAST(NULL AS STRING) AS nome_autor,
  CAST(NULL AS STRING) AS partido,
  uf,
  CAST(NULL AS INT) AS ordem_assinatura,
  'materias_autorias' AS id_lote
FROM workspace.pi_ii_silver.materias_autorias
WHERE id_materia IS NOT NULL AND id_senador IS NOT NULL
  AND (registro_valido_leg57 IS NULL OR registro_valido_leg57 = true);

-- 17 STATUS_ATUAL — situação de tramitação (Câmara)
-- PK(casa, id_proposicao); data_hora?; sigla_orgao?; descricao_tramitacao?; descricao_situacao?; apreciacao?; id_lote
CREATE OR REPLACE TABLE workspace.pi_ii_gold.status_atual AS
SELECT
  'CAMARA' AS casa,
  CAST(id_proposicao AS STRING) AS id_proposicao,
  ultimo_status_data_hora AS data_hora,
  ultimo_status_sigla_orgao AS sigla_orgao,
  ultimo_status_descricao_tramitacao AS descricao_tramitacao,
  ultimo_status_descricao_situacao AS descricao_situacao,
  ultimo_status_apreciacao AS apreciacao,
  'proposicoes' AS id_lote
FROM workspace.pi_ii_silver.proposicoes
WHERE id_proposicao IS NOT NULL
  AND (ultimo_status_data_hora IS NOT NULL
       OR ultimo_status_sigla_orgao IS NOT NULL
       OR ultimo_status_descricao_situacao IS NOT NULL);

-- 18 VOTACAO — votação de um órgão
-- PK(casa, id_votacao); data_votacao?; sigla_orgao?; aprovacao?; descricao?; votos_sim?; votos_nao?; votos_outros?; id_lote
CREATE OR REPLACE TABLE workspace.pi_ii_gold.votacao AS
SELECT
  'CAMARA' AS casa,
  id_votacao,
  data_votacao,
  sigla_orgao,
  CASE WHEN aprovacao = 1 THEN TRUE ELSE FALSE END AS aprovacao,
  descricao,
  votos_sim,
  votos_nao,
  votos_outros,
  'votacoes_camara' AS id_lote
FROM workspace.pi_ii_silver.votacoes_camara
WHERE id_votacao IS NOT NULL
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_votacao AS STRING) AS id_votacao,
  data_votacao,
  CAST(NULL AS STRING) AS sigla_orgao,
  CASE WHEN LOWER(resultado) LIKE '%aprovad%' THEN TRUE ELSE FALSE END AS aprovacao,
  descricao,
  votos_sim,
  votos_nao,
  votos_abstencao AS votos_outros,
  'votacoes_senado' AS id_lote
FROM workspace.pi_ii_silver.votacoes_senado
WHERE id_votacao IS NOT NULL;

In [0]:
%sql
-- ============================================================
-- TABELAS 19-22: Votação (cont.) e Total Publicado
-- ============================================================

-- 19 VOTACAO_PROPOSICAO — ligação votação ↔ proposição
-- PK(casa, id_votacao, id_proposicao); id_lote
-- FK(casa, id_votacao) → VOTACAO; FK(casa, id_proposicao) → PROPOSICAO
CREATE OR REPLACE TABLE workspace.pi_ii_gold.votacao_proposicao AS
SELECT
  'CAMARA' AS casa,
  id_votacao,
  CAST(id_proposicao AS STRING) AS id_proposicao,
  'votacoes_camara' AS id_lote
FROM workspace.pi_ii_silver.votacoes_camara
WHERE id_votacao IS NOT NULL AND id_proposicao IS NOT NULL
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_votacao AS STRING) AS id_votacao,
  CAST(id_materia AS STRING) AS id_proposicao,
  'votacoes_senado' AS id_lote
FROM workspace.pi_ii_silver.votacoes_senado
WHERE id_votacao IS NOT NULL AND id_materia IS NOT NULL;

-- 20 TIPO_VOTO — catálogo de tipos de voto
-- PK(casa, codigo_voto); descricao?
CREATE OR REPLACE TABLE workspace.pi_ii_gold.tipo_voto AS
SELECT
  'CAMARA' AS casa,
  voto AS codigo_voto,
  voto AS descricao
FROM (
  SELECT DISTINCT voto
  FROM workspace.pi_ii_silver.votos_camara
  WHERE voto IS NOT NULL AND voto != ''
)
UNION ALL
SELECT
  'SENADO' AS casa,
  voto AS codigo_voto,
  MAX(descricao_voto) AS descricao
FROM workspace.pi_ii_silver.votos_senado
WHERE voto IS NOT NULL AND voto != ''
GROUP BY voto;

-- 21 VOTO — voto individual
-- PK(casa, id_votacao, id_parlamentar); codigo_voto?; id_lote
-- FK(casa, id_votacao) → VOTACAO; FK(casa, id_parlamentar) → PARLAMENTAR
CREATE OR REPLACE TABLE workspace.pi_ii_gold.voto AS
SELECT
  'CAMARA' AS casa,
  id_votacao,
  CAST(id_deputado AS STRING) AS id_parlamentar,
  voto AS codigo_voto,
  'votos_camara' AS id_lote
FROM workspace.pi_ii_silver.votos_camara
WHERE id_votacao IS NOT NULL AND id_deputado IS NOT NULL
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_votacao AS STRING) AS id_votacao,
  CAST(id_senador AS STRING) AS id_parlamentar,
  voto AS codigo_voto,
  'votos_senado' AS id_lote
FROM workspace.pi_ii_silver.votos_senado
WHERE id_votacao IS NOT NULL AND id_senador IS NOT NULL;

-- 22 TOTAL_PUBLICADO — total de proposições por ano
-- PK(casa, ano); total_proposicoes; id_lote
CREATE OR REPLACE TABLE workspace.pi_ii_gold.total_publicado AS
SELECT
  'CAMARA' AS casa,
  NULLIF(ano_proposicao, 0) AS ano,
  COUNT(*) AS total_proposicoes,
  'proposicoes' AS id_lote
FROM workspace.pi_ii_silver.proposicoes
WHERE ano_proposicao IS NOT NULL AND ano_proposicao != 0
GROUP BY ano_proposicao
UNION ALL
SELECT
  'SENADO' AS casa,
  NULLIF(ano_materia, 0) AS ano,
  COUNT(*) AS total_proposicoes,
  'materias' AS id_lote
FROM workspace.pi_ii_silver.materias
WHERE ano_materia IS NOT NULL AND ano_materia != 0
  AND (registro_valido_leg57 IS NULL OR registro_valido_leg57 = true)
GROUP BY ano_materia;

In [0]:
%sql
-- ============================================================
-- TABELAS 23-26: Despesas (TIPO_DESPESA, FORNECEDOR, DESPESA, VALOR_DESPESA)
-- ============================================================

-- 23 TIPO_DESPESA — catálogo de tipos de despesa
-- PK(casa, codigo_tipo); descricao
CREATE OR REPLACE TABLE workspace.pi_ii_gold.tipo_despesa AS
SELECT
  'CAMARA' AS casa,
  tipo_despesa AS codigo_tipo,
  tipo_despesa AS descricao
FROM (
  SELECT DISTINCT tipo_despesa
  FROM workspace.pi_ii_silver.despesas_camara
  WHERE tipo_despesa IS NOT NULL AND tipo_despesa != ''
)
UNION ALL
SELECT
  'SENADO' AS casa,
  tipo_despesa AS codigo_tipo,
  tipo_despesa AS descricao
FROM (
  SELECT DISTINCT tipo_despesa
  FROM workspace.pi_ii_silver.despesas_senado
  WHERE tipo_despesa IS NOT NULL AND tipo_despesa != ''
);

-- 24 FORNECEDOR — fornecedor de despesa
-- PK(casa, cnpj_cpf); nome; id_lote
CREATE OR REPLACE TABLE workspace.pi_ii_gold.fornecedor AS
SELECT
  'CAMARA' AS casa,
  cnpj_cpf_fornecedor AS cnpj_cpf,
  MAX(fornecedor) AS nome,
  'despesas_camara' AS id_lote
FROM workspace.pi_ii_silver.despesas_camara
WHERE cnpj_cpf_fornecedor IS NOT NULL AND cnpj_cpf_fornecedor != ''
GROUP BY cnpj_cpf_fornecedor
UNION ALL
SELECT
  'SENADO' AS casa,
  cnpj_cpf_fornecedor AS cnpj_cpf,
  MAX(fornecedor) AS nome,
  'despesas_senado' AS id_lote
FROM workspace.pi_ii_silver.despesas_senado
WHERE cnpj_cpf_fornecedor IS NOT NULL AND cnpj_cpf_fornecedor != ''
GROUP BY cnpj_cpf_fornecedor;

-- 25 DESPESA — registro de despesa
-- PK(casa, id_despesa); id_parlamentar?; ano; mes; data_emissao?; codigo_tipo; cnpj_cpf?; id_lote
-- FK(casa, codigo_tipo) → TIPO_DESPESA; FK(casa, cnpj_cpf) → FORNECEDOR
CREATE OR REPLACE TABLE workspace.pi_ii_gold.despesa AS
SELECT
  'CAMARA' AS casa,
  CAST(id_documento AS STRING) AS id_despesa,
  CAST(id_deputado AS STRING) AS id_parlamentar,
  ano,
  mes,
  data_emissao,
  tipo_despesa AS codigo_tipo,
  cnpj_cpf_fornecedor AS cnpj_cpf,
  'despesas_camara' AS id_lote
FROM workspace.pi_ii_silver.despesas_camara
WHERE id_documento IS NOT NULL
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_despesa AS STRING) AS id_despesa,
  CAST(id_senador AS STRING) AS id_parlamentar,
  ano,
  mes,
  data_despesa AS data_emissao,
  tipo_despesa AS codigo_tipo,
  cnpj_cpf_fornecedor AS cnpj_cpf,
  'despesas_senado' AS id_lote
FROM workspace.pi_ii_silver.despesas_senado
WHERE id_despesa IS NOT NULL;

-- 26 VALOR_DESPESA — valores monetários da despesa
-- PK(casa, id_despesa); valor_documento?; valor_glosa?; valor_liquido?; id_lote
-- FK(casa, id_despesa) → DESPESA
CREATE OR REPLACE TABLE workspace.pi_ii_gold.valor_despesa AS
SELECT
  'CAMARA' AS casa,
  CAST(id_documento AS STRING) AS id_despesa,
  valor_documento,
  valor_glosa,
  valor_liquido,
  'despesas_camara' AS id_lote
FROM workspace.pi_ii_silver.despesas_camara
WHERE id_documento IS NOT NULL
UNION ALL
SELECT
  'SENADO' AS casa,
  CAST(id_despesa AS STRING) AS id_despesa,
  CAST(NULL AS DECIMAL(12,2)) AS valor_documento,
  CAST(NULL AS DECIMAL(12,2)) AS valor_glosa,
  valor_reembolsado AS valor_liquido,
  'despesas_senado' AS id_lote
FROM workspace.pi_ii_silver.despesas_senado
WHERE id_despesa IS NOT NULL;

In [0]:
%sql
-- ============================================================
-- VERIFICAÇÃO: contagem de linhas por tabela gold
-- ============================================================
SELECT 'casa' AS tabela, COUNT(*) AS linhas FROM workspace.pi_ii_gold.casa
UNION ALL SELECT 'uf', COUNT(*) FROM workspace.pi_ii_gold.uf
UNION ALL SELECT 'legislatura', COUNT(*) FROM workspace.pi_ii_gold.legislatura
UNION ALL SELECT 'partido', COUNT(*) FROM workspace.pi_ii_gold.partido
UNION ALL SELECT 'lote', COUNT(*) FROM workspace.pi_ii_gold.lote
UNION ALL SELECT 'causa_afastamento', COUNT(*) FROM workspace.pi_ii_gold.causa_afastamento
UNION ALL SELECT 'parlamentar', COUNT(*) FROM workspace.pi_ii_gold.parlamentar
UNION ALL SELECT 'identificador_parlamentar', COUNT(*) FROM workspace.pi_ii_gold.identificador_parlamentar
UNION ALL SELECT 'cadastro_observado', COUNT(*) FROM workspace.pi_ii_gold.cadastro_observado
UNION ALL SELECT 'parlamentar_legislatura', COUNT(*) FROM workspace.pi_ii_gold.parlamentar_legislatura
UNION ALL SELECT 'mandato', COUNT(*) FROM workspace.pi_ii_gold.mandato
UNION ALL SELECT 'vinculo_mandato', COUNT(*) FROM workspace.pi_ii_gold.vinculo_mandato
UNION ALL SELECT 'exercicio', COUNT(*) FROM workspace.pi_ii_gold.exercicio
UNION ALL SELECT 'tipo_proposicao', COUNT(*) FROM workspace.pi_ii_gold.tipo_proposicao
UNION ALL SELECT 'proposicao', COUNT(*) FROM workspace.pi_ii_gold.proposicao
UNION ALL SELECT 'autoria', COUNT(*) FROM workspace.pi_ii_gold.autoria
UNION ALL SELECT 'status_atual', COUNT(*) FROM workspace.pi_ii_gold.status_atual
UNION ALL SELECT 'votacao', COUNT(*) FROM workspace.pi_ii_gold.votacao
UNION ALL SELECT 'votacao_proposicao', COUNT(*) FROM workspace.pi_ii_gold.votacao_proposicao
UNION ALL SELECT 'tipo_voto', COUNT(*) FROM workspace.pi_ii_gold.tipo_voto
UNION ALL SELECT 'voto', COUNT(*) FROM workspace.pi_ii_gold.voto
UNION ALL SELECT 'total_publicado', COUNT(*) FROM workspace.pi_ii_gold.total_publicado
UNION ALL SELECT 'tipo_despesa', COUNT(*) FROM workspace.pi_ii_gold.tipo_despesa
UNION ALL SELECT 'fornecedor', COUNT(*) FROM workspace.pi_ii_gold.fornecedor
UNION ALL SELECT 'despesa', COUNT(*) FROM workspace.pi_ii_gold.despesa
UNION ALL SELECT 'valor_despesa', COUNT(*) FROM workspace.pi_ii_gold.valor_despesa
ORDER BY tabela;